# Homework 4 - Question 1: Differential Privacy in Deep Learning

**Course:** Security and Privacy in Machine Learning

**Student Name:** [Enter Name]

**Student ID:** [Enter ID]

---

## Overview
In this advanced assignment, you will go beyond basic DP implementation. Unlike standard tutorials that treat the optimizer as a black box, here you will:

1.  **Analyze Privacy Accounting:** Compare the theoretical bounds of Basic Composition vs. Rényi Differential Privacy (RDP).
2.  **Engineer for Speed:** Implement vectorized per-sample gradients using `torch.func` (formerly functorch) and benchmark it against naive loops.
3.  **Ensure Convergence:** Use **DP Fine-Tuning** on a pre-trained ResNet (instead of training from scratch) to observe actual utility-privacy tradeoffs.
4.  **Conduct Hyperparameter Studies:** Systematically analyze the effect of **Epsilon ($\epsilon$)** and **Clipping Norm ($C$)** on model performance.
5.  **Attack Your Model:** Perform a comparative **Membership Inference Attack (MIA)** to demonstrate the defense mechanism in action.
6.  **(Optional) Federated DP:** Implement User-Level Differential Privacy in a simulated Federated Learning setting.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision.transforms as transforms
import torchvision.datasets as datasets
import numpy as np
import matplotlib.pyplot as plt
from torch.func import functional_call, vmap, grad
import time
from opacus.accountants import RDPAccountant
from opacus.accountants.utils import get_noise_multiplier
import copy

# Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## Part 1: Advanced Privacy Accounting
Standard composition theorems are often too loose for Deep Learning. 

**The Task:**
1.  Calculate the noise $\sigma$ required for a target $\epsilon$ using **Rényi Differential Privacy (RDP)**.
2.  Visualize the accumulation of $\epsilon$ over epochs to understand why RDP is necessary.

In [ ]:
def compare_accountants(target_epsilon, target_delta, sample_rate, epochs):
    """
    Calculates the required noise multiplier and visualizes privacy budget consumption.
    """
    # TODO: 1. Use `get_noise_multiplier` from Opacus to find the required sigma
    sigma_rdp = ...
    
    print(f"Required Noise Multiplier (RDP): {sigma_rdp:.4f}")


    # TODO: 2. Visualize RDP accumulation
    # Instantiate an RDPAccountant()
    # Loop through epochs, call accountant.step(), and track epsilon history.

    # Plot eps_history vs epochs
    plt.plot(..., ...)
    plt.show()
    
    return 0.0 # Return the calculated sigma

### Question 1: Accounting Analysis
Observe the plot you generated above. How does the privacy budget $\epsilon$ grow with respect to the number of epochs (or steps) in the RDP accounting method? If we had used Basic Composition (where $\epsilon_{total} = \sum \epsilon_i$), how would the shape of the curve differ?

**Your Answer:**
[Double click to edit this markdown cell and write your answer here]

## Part 2: Engineering Efficient DP-SGD
Computing gradients for individual samples is the computational bottleneck of DP-SGD. 

**The Task:**
Implement `ManualDPSGD` using `torch.func.vmap`. You must **not** use a for-loop over the batch dimension in `compute_per_sample_gradients`.

In [ ]:
class ManualDPSGD:
    def __init__(self, model, lr, noise_multiplier, max_grad_norm):
        self.model = model
        self.noise_multiplier = noise_multiplier
        self.max_grad_norm = max_grad_norm
        self.params = [p for p in model.parameters() if p.requires_grad]
        self.base_optimizer = optim.SGD(self.params, lr=lr, momentum=0)

    def compute_per_sample_gradients(self, data, targets, loss_fn):
        """
        Computes per-sample gradients efficiently using torch.func.

         1. Define a stateless function `compute_loss_stateless(params, data, target)`
            Hint: Use `functional_call` to apply params to self.model
         2. Use `grad` to get gradients of that function.
         3. Use `vmap` to apply it over the batch dimension (in_dims).
        """

        # TODO: Implement the functional transform logic.

        params_dict = {k: v for k, v in self.model.named_parameters() if v.requires_grad}
        
        return per_sample_grads

    def clip_and_noise(self, per_sample_grads):
        """
        1. Calculates the L2 norm of gradients for EACH sample (Global Norm).
        2. Clips them to self.max_grad_norm.
        3. Sums them up.
        4. Adds Gaussian noise.
        """
        # TODO: Flatten grads per sample to compute the norm
        # TODO: Clip based on norm (use torch.clamp or min)
        # TODO: Aggregate (sum) and add Noise

        return summed_noisy_grads

    def step(self, data, targets, loss_fn):
        self.base_optimizer.zero_grad()
        
        # 1. Per-sample Grads
        per_sample_grads = self.compute_per_sample_gradients(data, targets, loss_fn)
        
        # 2. Clip & Noise
        final_grads = self.clip_and_noise(per_sample_grads)
        
        # 3. Assign grads back to model parameters
        for name, param in self.model.named_parameters():
            if param.requires_grad:
                # Important: Divide by batch size here if your loss_fn was reduction='mean' 
                # or if standard SGD expects averaged gradients.
                param.grad = final_grads[name] / data.shape[0]
                
        # 4. Step
        self.base_optimizer.step()

### Benchmarking
Run the cell below to verify that your `vmap` implementation is faster than a naive loop. If your implementation is correct, `Vectorized Time` should be significantly lower.

In [ ]:
def run_benchmark(model, data, targets):
    print("Benchmarking Gradient Computation...")
    
    # Naive Loop (Baseline)
    start = time.time()
    criterion = nn.CrossEntropyLoss()
    model.zero_grad()
    for i in range(data.shape[0]):
        output = model(data[i:i+1])
        loss = criterion(output, targets[i:i+1])
        loss.backward()
        model.zero_grad()
    end = time.time()
    print(f"Naive Loop Time (Batch {data.shape[0]}): {end - start:.4f}s")
    
    # Vectorized (Your Implementation)
    dpsgd = ManualDPSGD(model, lr=0.1, noise_multiplier=1.0, max_grad_norm=1.0)
    start = time.time()
    try:
        _ = dpsgd.compute_per_sample_gradients(data, targets, nn.CrossEntropyLoss())
        end = time.time()
        print(f"Vectorized (vmap) Time (Batch {data.shape[0]}): {end - start:.4f}s")
    except Exception as e:
        print(f"Vectorized implementation failed: {e}")

In [ ]:
# Create dummy inputs for the benchmark (Batch Size 64, CIFAR-10 image size)
dummy_data = torch.randn(64, 3, 32, 32).to(device)
dummy_targets = torch.randint(0, 10, (64,)).to(device)

# Create a simple dummy model (We use a simple one here just to test the gradient speed)
dummy_model = nn.Sequential(
    nn.Conv2d(3, 16, 3),
    nn.Flatten(),
    nn.Linear(16 * 30 * 30, 10)
).to(device)

# Run the benchmark
run_benchmark(dummy_model, dummy_data, dummy_targets)

## Part 3: Model Setup (DP Fine-Tuning)

Training from scratch with DP is difficult due to gradient noise. We will use **Transfer Learning** by fine-tuning a pre-trained ResNet18.

**The Task:**
1. Load a pre-trained ResNet18.
2. **Freeze** all layers except the final classification head.
3. Replace the final layer to match CIFAR-10 classes (10 outputs).

In [ ]:
def get_pretrained_model_for_cifar():
    # TODO: Load ResNet18 with pretrained=True
    model = ...
    
    # TODO: Freeze all parameters (requires_grad = False)
    
    # TODO: Replace the final fully connected layer (model.fc)
    # Ensure the new layer has requires_grad = True
    
    return model.to(device)

def train_model(model, train_loader, optimizer, epochs, is_dp=True):
    model.train()
    # Fix for Fine-Tuning: Force BatchNorm to Eval mode to prevent privacy leakage
    for module in model.modules():
        if isinstance(module, nn.BatchNorm2d):
            module.eval()
            
    criterion = nn.CrossEntropyLoss()
    
    for epoch in range(epochs):
        correct = 0
        total = 0
        
        for data, targets in train_loader:
            data, targets = data.to(device), targets.to(device)
            
            if is_dp:
                optimizer.step(data, targets, criterion)
            else:
                optimizer.zero_grad()
                output = model(data)
                loss = criterion(output, targets)
                loss.backward()
                optimizer.step()
                
            # Simple accuracy tracking
            with torch.no_grad():
                outputs = model(data)
                _, predicted = outputs.max(1)
                total += targets.size(0)
                correct += predicted.eq(targets).sum().item()
        
        print(f"Epoch {epoch+1}/{epochs} | Acc: {100.*correct/total:.2f}%")
    return 100.*correct/total

## Part 4: Hyperparameter Studies

In Differential Privacy, performance is highly sensitive to hyperparameters. You will now conduct two experiments to understand these trade-offs.

### 4.1. Effect of Privacy Budget ($\epsilon$)
**Goal:** Fix the Clipping Norm ($C=1.0$) and vary the target $\epsilon$. Observe how accuracy improves as you relax the privacy constraint (higher $\epsilon$).

**Task:**
1. Iterate through `epsilons = [0.5, 1.0, 3.0, 8.0]`.
2. For each epsilon, calculate the required $\sigma$, train a new model, and record the final accuracy.
3. Plot **Accuracy vs. Epsilon**.

In [ ]:
# Common Constants
# TODO: Adjust hyperparameters
BATCH_SIZE = ...
EPOCHS = ...
TARGET_DELTA = 1e-5

# Data Loading
transform_train = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])
trainset = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_train)
subset_indices = list(range(0, 5000)) # Train on 5000 images
train_subset = torch.utils.data.Subset(trainset, subset_indices)
train_loader = DataLoader(train_subset, batch_size=BATCH_SIZE, shuffle=True)
testset = datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_train)
test_loader = DataLoader(testset, batch_size=BATCH_SIZE, shuffle=False)
sample_rate = BATCH_SIZE / len(train_subset)

# --- Experiment 1: Varying Epsilon ---
epsilons = [0.5, 1.0, 3.0, 8.0]
accuracies_eps = []
dp_model = None
best_acc = 0

print("--- Running Epsilon Sweep ---")
for eps in epsilons:
    print(f"Training with Epsilon = {eps}...")
    
    # TODO: Calculate Sigma for this specific Epsilon
    sigma = ...
    
    # TODO: Train Model and compute the final accuracy
    
    # TODO: Save a model for Part 5 (one choice is best model)

    pass # remove this pass when implementing


# Plot Accuracy vs Epsilon
plt.figure()
plt.plot(epsilons, accuracies_eps, marker='o')
plt.xlabel('Privacy Budget (Epsilon)')
plt.ylabel('Final Accuracy (%)')
plt.title('Utility vs. Privacy Trade-off')
plt.grid(True)
plt.show()

### 4.2. Effect of Clipping Norm ($C$)
**Goal:** Fix the Privacy Budget ($\\epsilon=3.0$) and vary the Clipping Norm $C$. 
Recall that Noise $\propto \sigma \cdot C$. 
* **Small $C$:** Low noise variance, but high bias (gradients are crushed).
* **Large $C$:** Low bias, but massive noise variance.

**Task:**
1. Iterate through `clip_norms =  [0.01, 0.05, 0.1, 0.5, 0.7, 1, 1.5, 2, 2.5, 4, 7, 10]`.
2. Train a new model for each $C$ (keeping $\epsilon=3.0$ fixed).
3. Plot **Accuracy vs. Clipping Norm**.

In [ ]:
# --- Experiment 2: Varying Clip Norm ---
clip_norms = [0.01, 0.05, 0.1, 0.5, 0.7, 1, 1.5, 2, 2.5, 4, 7, 10]
accuracies_clip = []
FIXED_EPSILON = 3.0

# Calculate sigma ONCE since Epsilon and Epochs are fixed for this experiment
sigma_fixed = compare_accountants(FIXED_EPSILON, TARGET_DELTA, sample_rate, EPOCHS)

print("\n--- Running Clipping Norm Sweep ---")
for C in clip_norms:
    print(f"Training with Clip Norm C = {C}...")
    
    # TODO: Train Model and compute the final accuracy

    pass

# Plot Accuracy vs Clip Norm
plt.figure()
plt.plot(clip_norms, accuracies_clip, marker='o', color='orange')
plt.xscale('log') # Log scale helps visualize orders of magnitude
plt.xlabel('Clipping Norm (C)')
plt.ylabel('Final Accuracy (%)')
plt.title('Bias-Variance Trade-off (Clipping Norm)')
plt.grid(True)
plt.show()

### Question 2: Hyperparameter Analysis
1. **Epsilon Analysis:** Did accuracy increase monotonically with $\epsilon$? At what $\epsilon$ value did utility start to plateau (diminishing returns)?
2. **Clipping Norm Analysis:** Based on your second plot, is there a "sweet spot" for $C$? Explain why performance degrades if $C$ is too small (e.g., 0.1) versus too large (e.g., 10.0).

**Your Answer:**
[Double click to edit this markdown cell and write your answer here]

## Part 5: (Optional) Membership Inference Attack
Implement a **Loss-Threshold Attack** to demonstrate the privacy benefits of your DP model.
Use one of the **DP models** from your experiments above and compare it against a non-private baseline.

**The Task:**
1. Train a Non-DP Baseline Model (Standard SGD).
2. Calculate the CrossEntropyLoss for every sample in the Train Set and Test Set for both models.
3. Compute the ROC Curve assuming that `Loss(Member) < Loss(Non-Member)`.
4. Compare the AUC of the Non-DP model (High Risk) vs. the DP model (Low Risk).

In [ ]:
# 1. Train Non-DP Baseline (if not already done)
print("--- Training Non-DP Baseline ---")
base_model = get_pretrained_model_for_cifar()
base_optimizer = optim.SGD(filter(lambda p: p.requires_grad, base_model.parameters()), lr=0.01)
train_model(base_model, train_loader, base_optimizer, EPOCHS, is_dp=False)

# 2. Attack Implementation
def compute_losses(model, loader):
    model.eval()
    criterion = nn.CrossEntropyLoss(reduction='none')
    losses = []
    # TODO: Iterate over loader, compute loss per sample, append to list
    return np.array(losses)

def run_mia_analysis(model, name="Model"):
    print(f"Running MIA on {name}...")
    train_losses = compute_losses(model, train_loader)
    test_losses = compute_losses(model, test_loader)
    
    # TODO: Implement ROC Curve Calculation
    '''
     1. Create labels (1 for Member/Train, 0 for Non-Member/Test)
     2. Use scores = -losses (Since lower loss = higher likelihood of membership)
     3. Use sklearn.metrics.roc_curve and auc
    '''

    plt.plot(fpr, tpr, label=f'{name} (AUC = {roc_auc:.2f})')
    print(f"{name} AUC: {roc_auc:.4f}")


# Plot True Positive Rate vs False Positive Rate
plt.figure(figsize=(8, 6))
run_mia_analysis(base_model, "Non-DP Baseline")
if dp_model is not None:
    run_mia_analysis(dp_model, "Best DP Model")
plt.plot([0, 1], [0, 1], 'k--', label='Random Guess')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Membership Inference Attack Success')
plt.legend()
plt.show()

### Question 3: Attack Analysis
Compare the AUC scores of the Non-DP Model and the DP-SGD Model. 
1. What does an AUC of ~0.5 (for the DP model) indicate about the adversary's ability to infer membership?
2. Why does the Non-DP model likely have a much higher AUC, even if its test accuracy is similar?

**Your Answer:**
[Double click to edit this markdown cell and write your answer here]

## Part 6: (Optional) Federated Learning with Differential Privacy

In standard DP-SGD (Part 2), we clipped the gradient of **each sample** (Item-Level DP).
In Federated Learning, we want to protect the contribution of **each client** (User-Level DP). A client might have multiple data points.

**The Protocol (DP-FedAvg):**
1.  **Broadcast:** Server sends global model $\theta_t$ to selected clients.
2.  **Local Training:** Client $k$ trains on their local dataset $D_k$ to get $\theta_k$.
3.  **Compute Update:** $\Delta_k = \theta_k - \theta_t$.
4.  **Clip Update:** Scale $\Delta_k$ such that $||\Delta_k||_2 \le C$.
5.  **Aggregate & Noise:** $\Delta_{global} = \sum \Delta_{clipped} + \mathcal{N}(0, \sigma^2 C^2)$.
6.  **Update:** $\theta_{t+1} = \theta_t + \eta_{server} \cdot \Delta_{global}$.

**Task:** Implement the `server_aggregate` function to perform Clipping and Noising on client updates.

## Part 6.1: The Federated Client
In Federated Learning, the "Client" represents a single user or device (e.g., a smartphone).

The `SimpleFLClient` class handles **Local Training**. Unlike standard training, it:
1.  **Receives a Global Model:** Starts training from the server's current model state.
2.  **Freezes Batch Normalization:** We switch Batch Norm layers to `eval()` mode. If we didn't, the model would learn the running statistics (mean/variance) of the user's private data, which could leak information to the server.
3.  **Maintains State:** We use a persistent data iterator (`self.data_iter`) to ensure that if we call `train()` multiple times, the client continues through their dataset rather than restarting at the first batch every time.

In [ ]:
class SimpleFLClient:
    def __init__(self, data_loader):
        self.data_loader = data_loader
        # Helper: Create a persistent iterator to ensure we progress through data
        self.data_iter = iter(self.data_loader)

    def train(self, starting_model, steps=5, lr=0.01):
        """
        Trains the model locally for a specified number of steps.
        Args:
            starting_model: The global model sent by the server.
            steps: Number of local training steps (batches).
            lr: Learning rate.
        """
        # Local SGD Training (Standard, no DP per step)
        local_model = copy.deepcopy(starting_model)
        local_model.train()

        # Freeze Batch Norm to avoid privacy leakage via running stats
        for module in local_model.modules():
            if isinstance(module, nn.BatchNorm2d):
                module.eval()

        # Only optimize trainable parameters (the head)
        optimizer = optim.SGD(filter(lambda p: p.requires_grad, local_model.parameters()), lr=lr)
        criterion = nn.CrossEntropyLoss()

        for i in range(steps):
            try:
                # Use the persistent iterator
                data, target = next(self.data_iter)
            except StopIteration:
                # Restart iterator if data runs out
                self.data_iter = iter(self.data_loader)
                data, target = next(self.data_iter)

            data, target = data.to(device), target.to(device)
            optimizer.zero_grad()
            output = local_model(data)
            loss = criterion(output, target)
            loss.backward()
            optimizer.step()

        return local_model

## Part 6.2: Server Aggregation (DP-FedAvg)
This function implements the core of **User-Level Differential Privacy**.

Unlike Part 2 (where we clipped *per-sample* gradients), here we clip the **entire model update** from a single user. This protects the user's participation in the training process.

**The Protocol:**
1.  **Compute Delta ($\Delta$):** Calculate how much the client changed the model (`Local Model - Global Model`).
2.  **Filter Parameters:** We **only** apply noise to the trainable classification head. We must **ignore** the frozen pre-trained backbone (ResNet layers), or else the noise would destroy the pre-trained features.
3.  **Clip:** Calculate the L2 norm of the update. If it exceeds `clip_norm`, scale it down.
4.  **Noise:** Add Gaussian noise to the sum of clipped updates.
5.  **Average:** Divide by the number of clients to get the average update.

In [ ]:
def server_aggregate(server_model, client_models, noise_multiplier, clip_norm):
    """
    Aggregates client models using DP-FedAvg logic:
    1. Calculate Update (Delta)
    2. Clip Delta (User-Level DP)
    3. Add Noise
    4. Update Global Model
    """
    global_dict = server_model.state_dict()

    # Identify which parameters are trainable.
    # We MUST NOT add noise to frozen parameters (the backbone).
    trainable_keys = [name for name, param in server_model.named_parameters() if param.requires_grad]

    # Initialize accumulated update to zeros ONLY for trainable keys
    accumulated_update = {name: torch.zeros_like(global_dict[name], dtype=torch.float) for name in trainable_keys}

    num_clients = len(client_models)

    for client_model in client_models:
        client_dict = client_model.state_dict()

        # TODO: 1. Compute Update vector (Delta) for this client
        delta = ...
        # Store deltas temporarily. Only process 'trainable_keys'.

        # TODO: 2. Calculate L2 Norm of this entire Delta vector
        # Hint: Flatten all deltas (for trainable keys) into one vector to get the global norm.

        # TODO: 3. Compute Scaling Factor
        factor = ...

        # TODO: 4. Clip and Add to Accumulator
        accumulated_update[name] += ...
        pass

    # TODO: 5. Add Gaussian Noise to the Accumulator
    # For each parameter in accumulated_update, add noise with std = noise_multiplier * clip_norm

    # Apply update to global model
    new_global_dict = global_dict.copy()
    for name in trainable_keys:
        # TODO: 6. Average the noisy sum and update global parameters
        pass

    server_model.load_state_dict(new_global_dict)
    return server_model

## Part 6.3: Running the Simulation
We will now simulate a Federated Learning scenario with **User-Level DP**.

**Hyperparameters:**
* `NUM_CLIENTS = 5`: A small number of clients for demonstration.
* `FL_ROUNDS = ...`: The number of times the server aggregates updates.
* `LOCAL_STEPS = ...`: The number of batches a client trains on locally before sending an update. Increasing this is crucial for convergence in FL.
* `FL_NOISE = ...`: The amount of privacy noise added.

**Instructions:**
1.  **Split Data:** We split the training subset into chunks, one for each client.
2.  **Train:** In every round, each client downloads the global model, improves it on their private data, and sends the update back.
3.  **Aggregate:** The server combines these updates securely using the `server_aggregate` function you implemented.

In [ ]:
# --- Simulation Configuration ---
# Optimized hyperparameters for better accuracy
NUM_CLIENTS = 5
FL_ROUNDS = ...
FL_NOISE = ...
FL_CLIP = ...
LOCAL_STEPS = ...

print(f"\n--- Running FL Simulation (Clients={NUM_CLIENTS}, Rounds={FL_ROUNDS}) ---")

# 1. Setup Data and Clients
# Use a larger subset (e.g. 10,000 or more) for better stability
subset_indices = list(range(0, 10000))
# Ensure 'trainset' matches your variable name from previous cells
train_subset = torch.utils.data.Subset(trainset, subset_indices)

# Split train_subset into NUM_CLIENTS chunks and create SimpleFLClient instances
lengths = [len(train_subset) // NUM_CLIENTS] * NUM_CLIENTS
client_chunks = torch.utils.data.random_split(train_subset, lengths)
clients = ...

# 2. Initialize Global Model
global_model = get_pretrained_model_for_cifar()
fl_accuracies = []

# 3. Federated Learning Loop
for round_idx in range(FL_ROUNDS):
    # TODO: A. Local Training
    # Iterate through clients, call client.train(global_model, steps=LOCAL_STEPS)
    # Store resulting local models in a list.

    # TODO: B. DP Aggregation
    # Call server_aggregate() to update the global_model

    # TODO: C. Evaluation
    # Evaluate global_model on test_loader and append accuracy to fl_accuracies
    correct = ...

    # print(f"Round {round_idx+1}/{FL_ROUNDS} | Global Acc: {acc:.2f}%")
    pass

# Plot Results
plt.figure(figsize=(8, 5))
plt.plot(range(1, FL_ROUNDS + 1), fl_accuracies, marker='o', label='DP-FedAvg')
plt.xlabel('Federated Rounds')
plt.ylabel('Test Accuracy (%)')
plt.title('Federated Learning with User-Level DP')
plt.grid(True)
plt.legend()
plt.show()

## Part 6.4: Critical Analysis & Discussion

Now that you have implemented and observed both **Centralized DP-SGD** (Item-Level Privacy) and **Federated Learning with DP** (User-Level Privacy), answer the following questions to compare the two approaches.

### Question 4: User-Level vs. Item-Level Privacy
1.  **Scope of Protection:** In Part 2 (Centralized DP-SGD), we clipped the gradient of every *sample*. In Part 6 (Federated), we clipped the update of every *client*.
    * If a specific client has a unique dataset (e.g., photos of a rare object), which method (Centralized or Federated) offers better protection for that *specific* object? Why?
    * Which method better conceals the fact that the client *participated* in the training at all?

2.  **Utility & Convergence:**
    * Compare the convergence speed and final accuracy of the Federated model vs. the Centralized DP model.
    * Why is the "Noise-to-Signal" ratio typically worse in Federated Learning when $N$ (number of clients) is small (e.g., 5), compared to Centralized learning where $N$ (number of samples) is large (e.g., 50,000)?

3.  **The "Straggler" Problem (Thought Experiment):**
    * In our simulation, all clients computed updates instantly. In a real-world scenario with millions of phones, some devices are slow or drop out. If we drop the bottom 10% of updates, how might that bias the model? (Hint: Consider if the "slow" devices are older phones owned by a specific demographic).

**Your Answer:**
[Double click to edit this markdown cell and write your answer here]